# lung-clam on Google Colab

Runs the heavy parts of [lung-clam](https://github.com/amaansayydd/lung-clam) on a free Colab GPU:

1. **Extract**: download each slide from NCI Imaging Data Commons, segment tissue, cut 128 µm patches, encode them with ResNet50 and Phikon, delete the slide.
2. **Train**: CLAM and baselines, 5 patient-level folds each.
3. **Evaluate**: tables, figures, shortcut check, attention heatmaps.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.

Everything is saved to Google Drive (`MyDrive/lung_clam_data`, about 4–5 GB), and every step skips work that is already done. If Colab disconnects, reconnect and **Run all** again; it continues where it stopped.

## 1 · Settings

In [ ]:
STORAGE = "drive"          # "drive": survives disconnects (recommended) · "local": Colab disk only, zip at the end
DRIVE_FOLDER = "lung_clam_data"
ENCODERS = ["resnet50_trunc", "phikon"]   # add "uni2_h" only if your Hugging Face account has UNI2 access
RUN_EXTRACTION = True
EXPERIMENT_GROUPS = ["core", "ablations", "label_efficiency"]   # see scripts/run_experiments.py

## 2 · GPU, storage and code

In [ ]:
import os, shutil, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
assert gpu, "No GPU. Runtime → Change runtime type → T4 GPU, then run again."
print("GPU:", gpu)

if STORAGE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
else:
    DATA = "/content/lung_clam_data"
os.makedirs(DATA, exist_ok=True)
os.environ.update(LUNGCLAM_DATA=DATA, LUNGCLAM_RAW="/content/raw", LUNGCLAM_RESULTS=f"{DATA}/results",
                  PYTORCH_ENABLE_MPS_FALLBACK="1")
free = shutil.disk_usage(DATA).free / 1e9
print(f"data folder: {DATA}  ({free:.1f} GB free)")
if free < 6:
    print("⚠️  Less than 6 GB free here. Features need about 4–5 GB; free some space or set STORAGE = 'local'.")

In [ ]:
%cd /content
if not os.path.exists("/content/lung-clam"):
    !git clone -q --recurse-submodules https://github.com/amaansayydd/lung-clam.git
else:
    !cd /content/lung-clam && git pull -q && git submodule update -q --init
%cd /content/lung-clam
!git -C CLAM apply ../patches/0001-apple-mps-and-fp16-features.patch 2>/dev/null && echo "CLAM patched" || echo "CLAM patch already applied"
!pip install -q openslide-python openslide-bin idc-index "timm>=1.0.15" h5py future tabulate \
    "topk @ git+https://github.com/oval-group/smooth-topk.git@12c1645f187e2fa0c05f47bf1fe48864d4bd2707"
!python scripts/check_env.py

## 3 · Extract features

Downloads, patches and encodes all 570 slides (about 262 GB streamed; only a few slides on disk at a time). Prints one line per slide with a running ETA. Safe to interrupt and re-run.

In [ ]:
if RUN_EXTRACTION:
    !python scripts/run_stream.py --encoders {" ".join(ENCODERS)} --batch 4

In [ ]:
import pandas as pd
st = pd.read_csv(f"{DATA}/status.csv").drop_duplicates("slide_id", keep="last")
print(st.status.value_counts().to_string())
print(f"patches: {st.n_patches.sum():,.0f} total, {st.n_patches.median():,.0f} per slide (median)")
failed = st[st.status != "done"]
if len(failed):
    print("\nfailed slides (re-run the cell above to retry):")
    print(failed[["slide_id", "detail"]].to_string(index=False))

## 4 · Train and evaluate

Every experiment trains 5 folds; finished folds are skipped on re-run. The internal test set stays sealed: it is scored only by the final model, in section 6.

In [ ]:
!python scripts/run_experiments.py {" ".join(EXPERIMENT_GROUPS)}

In [ ]:
!python scripts/40_shortcut_baseline.py
!python scripts/30_evaluate.py
from IPython.display import Markdown, display
display(Markdown(open(f"{DATA}/results/tables/metrics.md").read()))

## 5 · Figures and heatmaps

In [ ]:
!python scripts/60_figures.py --main clam_sb_phikon
!python scripts/50_heatmaps.py clam_sb_phikon
from IPython.display import Image, display
for f in ["model_comparison", "label_efficiency", "confusion_clam_sb_phikon", "roc_clam_sb_phikon"]:
    p = f"{DATA}/results/figures/{f}.png"
    if os.path.exists(p):
        display(Image(p, width=700))

## 6 · Final model and the sealed test set (run once, at the very end)

Trains the pre-registered final configuration with 3 seeds and scores the internal test set for the first and only time. Leave `UNSEAL = False` until every other decision is final.

In [ ]:
UNSEAL = False
if UNSEAL:
    !python scripts/20_train.py configs/experiments/final_clam_sb_phikon.yaml --with-test
    !python scripts/30_evaluate.py --unseal
    display(Markdown(open(f"{DATA}/results/tables/metrics.md").read()))

## 7 · Download the results (a few MB) to send back to the Mac

In [ ]:
shutil.make_archive("/content/lung_clam_results", "zip", f"{DATA}/results")
from google.colab import files
files.download("/content/lung_clam_results.zip")